# День 2. Контекст, семантические представления, графы понятий и генерация вопросов

**План на 4 пары по 90 минут.** День продолжает первый: от лексического сходства — к смысловым представлениям, структуре знаний и формированию вопросов и ответов из учебного материала.

Сквозной сценарий: взять текст главы по теории вероятностей/математической статистике, разбить на контекстные фрагменты, найти релевантный фрагмент, построить граф понятий и подготовить вопросы с ответами. Финальный этап — оценить качество ответов и близость автоматически сформулированного вопроса к исходному содержанию.

**Базовые библиотеки:** `scikit-learn`, `networkx`, `matplotlib`. Эмбеддинги на готовой модели — дополнительный блок, может потребовать загрузки модели из интернета.

## Пара 1. Контекст: от книги к фрагментам для поиска

### 1.1. Почему нельзя векторизовать книгу как один документ?

Если весь учебник превратить в один вектор, теряется локальный контекст: сложно понять, какой абзац определяет термин, формулирует теорему или описывает условие применения метода. Поэтому длинный документ разбивают на фрагменты (chunks), сохраняют метаданные и векторизуют каждый фрагмент отдельно.

В реальном генераторе тестов важно хранить источник, главу/раздел и номер фрагмента, чтобы ответ можно было проверить по исходному тексту.

In [ ]:
chapter = """
Случайная величина называется дискретной, если множество её возможных значений конечно или счётно.
Математическое ожидание дискретной случайной величины равно сумме произведений её значений на соответствующие вероятности.
Дисперсия характеризует разброс значений случайной величины относительно математического ожидания.
Для выборки выборочное среднее вычисляется как сумма наблюдений, делённая на объём выборки.
Несмещённая оценка дисперсии использует сумму квадратов отклонений от выборочного среднего и делитель n - 1.
Доверительный интервал задаёт диапазон значений параметра, согласующихся с данными при выбранном уровне доверия.
Нулевая гипотеза формулирует проверяемое предположение о распределении или параметре генеральной совокупности.
Уровень значимости — вероятность ошибки первого рода, то есть отклонения верной нулевой гипотезы.
Корреляция описывает направление и силу линейной связи между количественными переменными, но сама по себе не доказывает причинность.
"""

paragraphs = [p.strip() for p in chapter.strip().split("\n") if p.strip()]
for i, p in enumerate(paragraphs):
    print(f"[chunk {i}] {p}")

### 1.2. Разбиение с перекрытием

Фиксированные фрагменты по числу слов просты, но могут разрывать определение на границе. Перекрытие помогает сохранить контекст между соседними фрагментами. В более серьёзной системе границы лучше определять по абзацам, заголовкам или семантическим блокам.

In [ ]:
def chunk_words(text, size=18, overlap=5):
    words = text.split()
    if overlap >= size:
        raise ValueError("overlap должен быть меньше size")
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start+size]))
        if start + size >= len(words):
            break
        start += size - overlap
    return chunks

for i, chunk in enumerate(chunk_words(chapter, size=18, overlap=5)):
    print(f"Chunk {i}: {chunk}\n")

### 1.3. Retrieval: поиск контекста для вопроса

Retrieval — этап извлечения релевантных фрагментов. Сначала ищем контекст, затем на его основе формулируем ответ. Это отличается от генерации ответа только по памяти языковой модели: ответ должен быть обоснован найденным источником.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

chunks = paragraphs
retriever = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_chunks = retriever.fit_transform(chunks)

def retrieve(question, k=3):
    q = retriever.transform([question])
    scores = cosine_similarity(q, X_chunks).ravel()
    order = np.argsort(scores)[::-1][:k]
    return [(int(i), round(float(scores[i]), 3), chunks[i]) for i in order]

for result in retrieve("Как интерпретировать уровень значимости?"):
    print(result)

### Задания к паре 1

1. Разбейте один учебный раздел двумя способами: по предложениям и по фиксированному числу слов. Сравните фрагменты.
2. Почему ответ должен сохранять ссылку на источник?
3. Что хуже для поиска: слишком короткий фрагмент или слишком длинный? Ответ зависит от задачи — объясните.
4. Составьте пять запросов и отметьте, оказался ли правильный фрагмент в топ-1 и топ-3.

## Пара 2. Эмбеддинги и семантическая близость

### 2.1. Лексическая и семантическая близость

TF–IDF оценивает совпадение слов и n-грамм. Эмбеддинги кодируют текст плотным вектором, где близость может отражать семантические закономерности. Но модель может ошибаться на узкой терминологии, отрицаниях и математических условиях. Поэтому нужен тестовый набор пар с известной релевантностью.

In [ ]:
pairs = [
    ("разброс выборки", "дисперсия наблюдений", 1),
    ("вероятность не менее двух успехов", "вероятность двух или более успехов", 1),
    ("нулевая гипотеза не отвергается", "нулевая гипотеза отвергается", 0),
    ("доверительный интервал среднего", "выборочная медиана", 0),
    ("ошибка первого рода", "отклонение верной нулевой гипотезы", 1)
]
for a, b, label in pairs:
    score = cosine_similarity(
        retriever.transform([a]), retriever.transform([b])
    )[0, 0]
    print(f"label={label} TF-IDF={score:.3f} | {a} <> {b}")

### 2.2. Эмбеддинги готовой модели (дополнительно)

Следующий блок использует многоязычную модель Sentence Transformers. При первом запуске модель скачивается из интернета. Если в аудитории нет доступа к сети, пропустите блок и используйте TF–IDF. Перед практическим внедрением проверьте лицензию, размер модели и качество на предметном тестовом наборе.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    pair_texts = [x for a, b, _ in pairs for x in (a, b)]
    E = model.encode(pair_texts, normalize_embeddings=True)
    for i, (a, b, expected) in enumerate(pairs):
        score = float(E[2*i] @ E[2*i+1])
        print(f"expected={expected} embedding_cosine={score:.3f} | {a} <> {b}")
except ImportError:
    print("Для дополнительного блока: python -m pip install sentence-transformers")
except Exception as exc:
    print("Модель недоступна; можно продолжить с TF-IDF:", type(exc).__name__)

### 2.3. Метрика близости — не метрика качества

Косинусная близость — способ сравнить два вектора. Она не сообщает сама по себе, полезен ли найденный ответ. Для поиска проверяют, попадает ли правильный фрагмент в топ-k (Recall@k), а для пар «похожи/не похожи» можно оценивать качество классификации или ранжирования. Порог близости следует подбирать на размеченных примерах, а не назначать произвольно.

In [ ]:
# Простой тест: попадает ли ожидаемый фрагмент в топ-1 / топ-3?
queries = [
    ("Что характеризует дисперсия?", "Дисперсия характеризует разброс"),
    ("Что такое уровень значимости?", "Уровень значимости — вероятность"),
    ("Что означает корреляция?", "Корреляция описывает направление")
]
for question, expected_prefix in queries:
    results = retrieve(question, k=3)
    found = any(expected_prefix.lower() in text.lower() for _, _, text in results)
    print(question, "->", "ожидаемый фрагмент в топ-3:", found)
    for row in results[:2]:
        print(" ", row[:2], row[2][:100])

### Задания к паре 2

1. Создайте 10 пар перефразировок и 10 пар с противоположным смыслом.
2. Проверьте, не считает ли метрика близкими фразы с отрицанием.
3. Сравните TF–IDF и эмбеддинги на одних и тех же парах.
4. Объясните, почему нельзя оценивать качество системы только средним косинусным сходством.

## Пара 3. Граф понятий и онтология предметной области

### 3.1. Что добавляет граф?

Векторное представление описывает текст численно, но не задаёт явные отношения между понятиями. Граф знаний представляет понятия как узлы, а отношения — как рёбра: «дисперсия — мера разброса», «уровень значимости — вероятность ошибки первого рода». Онтология дополнительно задаёт типы понятий и допустимые отношения. В этом учебном примере граф небольшой и создан вручную; извлечение отношений из произвольной книги — отдельная сложная задача.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
triples = [
    ("Случайная величина", "имеет характеристику", "Математическое ожидание"),
    ("Случайная величина", "имеет характеристику", "Дисперсия"),
    ("Дисперсия", "характеризует", "Разброс"),
    ("Выборка", "используется для оценки", "Параметр генеральной совокупности"),
    ("Выборочное среднее", "оценивает", "Математическое ожидание"),
    ("Доверительный интервал", "оценивает", "Параметр генеральной совокупности"),
    ("Проверка гипотез", "использует", "Нулевая гипотеза"),
    ("Уровень значимости", "задаёт вероятность", "Ошибка первого рода"),
    ("Корреляция", "характеризует", "Линейная связь"),
    ("Корреляция", "не доказывает", "Причинность")
]
for source, relation, target in triples:
    G.add_edge(source, target, relation=relation)

pos = nx.spring_layout(G, seed=17, k=0.9)
plt.figure(figsize=(14, 9))
nx.draw_networkx_nodes(G, pos, node_size=2200)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=15, connectionstyle="arc3,rad=0.08")
edge_labels = {(u, v): d["relation"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
plt.axis("off")
plt.title("Фрагмент графа понятий по математической статистике")
plt.show()

### 3.2. Тройки «субъект — отношение — объект»

Граф можно хранить в виде троек (subject, relation, object). Такой формат удобно валидировать, экспортировать в CSV/JSON и использовать для построения визуализации. В реальной онтологии нужны определения классов, ограничения и идентификаторы терминов.

In [ ]:
print("Количество узлов:", G.number_of_nodes())
print("Количество отношений:", G.number_of_edges())
for u, v, data in G.edges(data=True):
    print({"subject": u, "relation": data["relation"], "object": v})

### 3.3. Использование графа для вопросов

По узлам и рёбрам можно строить шаблонные вопросы: «Что характеризует дисперсия?», «Какую вероятность задаёт уровень значимости?», «Что оценивает доверительный интервал?». Ответ из графа нужно проверять по исходному контексту: граф — структурированная модель знаний, а не доказательство истинности утверждения.

In [ ]:
def questions_from_graph(graph):
    generated = []
    for subject, target, data in graph.edges(data=True):
        relation = data["relation"]
        generated.append({
            "question": f"Что связывает понятие «{subject}» с понятием «{target}»?",
            "answer": f"{subject} — {relation} — {target}.",
            "source_type": "учебный граф"
        })
    return generated

for item in questions_from_graph(G)[:5]:
    print(item)

### Задания к паре 3

1. Добавьте в граф понятия «выборочная дисперсия», «несмещённая оценка», «объём выборки» и корректные отношения.
2. Найдите понятия, связанные с двумя и более другими понятиями. Как это помогает формировать вопросы?
3. Почему связь «корреляция — причинность» нельзя записать как «доказывает»?
4. Чем граф понятий отличается от списка ключевых слов и от TF–IDF-вектора?
5. Укажите, какие рёбра вы добавили по тексту источника, а какие — как экспертное знание.

## Пара 4. Генерация вопросов и ответов, проверка качества

### 4.1. Извлечение контекста и создание QA-примеров

Практический конвейер генератора тестов может состоять из шагов:
1. извлечь текст главы и метаданные источника;
2. разбить текст на контекстные фрагменты;
3. найти фрагмент, релевантный теме или запросу;
4. сформулировать вопрос по фрагменту;
5. получить ответ, опирающийся на этот же фрагмент;
6. проверить соответствие ответа источнику и устранить дубликаты.

В этой демонстрации вопросы строятся шаблонами и вручную заданными фактами. Это позволяет изучить логику и оценку, не притворяясь, что полноценная генеративная модель уже подключена.

In [ ]:
qa_source = [
    {
        "context": paragraphs[1],
        "question": "Как вычисляется математическое ожидание дискретной случайной величины?",
        "answer": "Как сумма произведений значений на соответствующие вероятности."
    },
    {
        "context": paragraphs[2],
        "question": "Что характеризует дисперсия?",
        "answer": "Разброс значений случайной величины относительно математического ожидания."
    },
    {
        "context": paragraphs[4],
        "question": "Какой делитель используется в несмещённой оценке дисперсии?",
        "answer": "n - 1."
    },
    {
        "context": paragraphs[7],
        "question": "Что такое ошибка первого рода?",
        "answer": "Отклонение верной нулевой гипотезы."
    }
]
for item in qa_source:
    print("Контекст:", item["context"])
    print("Вопрос:", item["question"])
    print("Ответ:", item["answer"], "\n")

### 4.2. Генерация вопроса из предложения: шаблонный базовый уровень

Шаблонные методы дают воспроизводимую базовую линию, но вопросы могут звучать однообразно и не проверять глубокое понимание. Для генерации с языковой моделью нужно дополнительно ограничить источник, потребовать краткий ответ и проверять фактическую опору в контексте.

In [ ]:
question_templates = [
    ("Что характеризует", "Дисперсия характеризует разброс значений случайной величины."),
    ("Что задаёт", "Уровень значимости задаёт вероятность ошибки первого рода."),
    ("Что описывает", "Корреляция описывает направление и силу линейной связи.")
]
for stem, sentence in question_templates:
    print("Контекст:", sentence)
    print("Вопрос:", stem + " ...?")
    print("Ожидаемый смысл ответа:", sentence)

### 4.3. Сравнение по близости и проверка ответа

Косинусное сходство вопроса и ответа не является универсальной метрикой качества QA. Сравнивать можно ответ с эталонным ответом, вопрос с исходным контекстом, либо найденный фрагмент с запросом — это разные задачи. Для оценки ответа полезны:
- семантическая близость к эталону как автоматический сигнал;
- проверка, подтверждается ли ответ контекстом;
- экспертная проверка корректности, полноты и однозначности;
- обнаружение дубликатов вопросов.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

reference_answers = [
    "Разброс значений случайной величины относительно математического ожидания.",
    "Вероятность отклонения верной нулевой гипотезы.",
    "Направление и сила линейной связи между количественными переменными."
]
candidate_answers = [
    "Дисперсия показывает, насколько значения разбросаны вокруг математического ожидания.",
    "Это вероятность ошибки первого рода.",
    "Корреляция описывает линейную связь между двумя переменными."
]
metric_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
A = metric_vectorizer.fit_transform(reference_answers + candidate_answers)
S = cosine_similarity(A[len(reference_answers):], A[:len(reference_answers)])
for i, row in enumerate(S):
    best = int(row.argmax())
    print(f"Кандидат {i+1}: лучший эталон {best+1}, TF-IDF cosine={row[best]:.3f}")
    print("  кандидат:", candidate_answers[i])
    print("  эталон:  ", reference_answers[best])

### 4.4. Финальный проект: мини-генератор тестов

Выберите небольшой учебный текст по теории вероятностей или матстатистике и реализуйте прототип:

1. хранение исходных фрагментов с указанием раздела;
2. поиск контекста по запросу через TF–IDF и косинусную близость;
3. граф понятий минимум из 10 узлов и 12 отношений;
4. генерация не менее 10 вопросов — шаблонных, по отношениям графа или вручную подготовленных по контексту;
5. ответы с обязательной ссылкой на исходный фрагмент;
6. сравнение кандидатов с эталонами по метрике близости;
7. ручная оценка корректности, полноты и соответствия источнику.

**Важно:** сходство текста не гарантирует истинность. Высокая метрика может быть у ответа, который содержит ошибку, пропускает условие или меняет отрицание.